In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys
import pandas as pd
import datetime as dt

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

from report_builder import ReportBuilder
from utils.merger import Merger

In [3]:
NOW = dt.datetime.now()

In [4]:
clients_path = "./all_clicnts.pkl"
orgs_path = "./all_orgs.pkl"
elv_plans_path = "./all_elv_plans.pkl"
cu_plans_path = "./all_cu_plans.pkl"

clients_df = pd.read_pickle(clients_path)
print(len(clients_df))
orgs_df = pd.read_pickle(orgs_path)
print(len(orgs_df))
elv_plans_df = pd.read_pickle(elv_plans_path)
print(len(elv_plans_df))
cu_plans_df = pd.read_pickle(cu_plans_path)
print(len(cu_plans_df))

2761
1598
5883
6175


In [5]:
merger = Merger(
    organization_df=orgs_df,
    elv_plan_df=elv_plans_df,
    client_df=clients_df,
    cu_plan_df=cu_plans_df
)

clickup_df = merger.merge_clickup()
elevate_df = merger.merge_elevate()


print(len(orgs_df))
print(len(clients_df))

1598
2761


In [6]:
necessary_columns = [
    "Plan Year Start Date (Month & Day)",
    "Plan Year End Date (Month & Day)",
    "Employer HSA Contributions",
    "Health FSA Limit",
    "Rollover/Carryover",
    "Rollover Max",
    "Carryover only through next Plan Year",
    "Carryover only if elect to participate for next Plan Year",
    "Limited Purpose Offered",
    "HSA Offered",
    "Health FSA Grace Period Offered",
    "Health FSA Grace Period Length",
    "Dependent FSA Grace Period Offered",
    "Dependent FSA Grace Period Length",
    "Runout Period Length",
    "Termed Runout Period Length"
]

In [7]:
"account_type.account_type" in elevate_df.columns

True

In [8]:
[c for c in elevate_df.columns if "auto" in c]

['plan_account_funding_config.is_auto_enrollment.is_auto_enrollment',
 'plan_account_funding_config.is_auto_enrollment.is_auto_enrollment_state']

In [9]:
elevate_df[['plan_account_funding_config.is_auto_enrollment.is_auto_enrollment',
 'plan_account_funding_config.is_auto_enrollment.is_auto_enrollment_state']]

,plan_account_funding_config.is_auto_enrollment.is_auto_enrollment,plan_account_funding_config.is_auto_enrollment.is_auto_enrollment_state
0,False,HIDDEN
1,False,HIDDEN
2,False,HIDDEN
3,True,MODIFIABLE
4,True,MODIFIABLE
...,...,...
5878,False,HIDDEN
5879,False,HIDDEN
5880,False,HIDDEN
5881,True,MODIFIABLE


In [10]:
elevate_df[
    (elevate_df['rmrcode'] == "RMRMNS") &
    (elevate_df['account_type.account_type'] == "HSA")
][[
    'elv_plan_id',
    'plan_year.valid_from',
    'plan_primary_config.max_election_amount_type.max_election_amount',
    'plan_primary_config.max_election_amount_type.max_election_amount_type'
]].sort_values(by="plan_year.valid_from", ascending= False)

,elv_plan_id,plan_year.valid_from,plan_primary_config.max_election_amount_type.max_election_amount,plan_primary_config.max_election_amount_type.max_election_amount_type
5791,45021,2025-01-01,8550.0,IRS_LIMIT
5786,34241,2024-01-01,8300.0,IRS_LIMIT


In [26]:
display(elevate_df['plan_account_funding_config.is_auto_enrollment.is_auto_enrollment'].value_counts())

plan_account_funding_config.is_auto_enrollment.is_auto_enrollment
False    3607
True     2154
Name: count, dtype: int64

In [39]:
clickup_df['cu_plan_status'].value_counts()

cu_plan_status
active         4831
terminated      438
setup           296
review          294
past             83
offboarding      65
prospect          1
Name: count, dtype: int64

In [36]:
[c for c in elevate_df.columns if "enroll" in c]


['plan_account_funding_config.is_auto_enrollment.is_auto_enrollment',
 'plan_account_funding_config.is_auto_enrollment.is_auto_enrollment_state']

In [38]:
[c for c in clickup_df.columns if "status" in c]

['cu_client_status',
 'status.id_x',
 'status.color_x',
 'status.type_x',
 'status.orderindex_x',
 'cu_plan_status',
 'status.id_y',
 'status.color_y',
 'status.type_y',
 'status.orderindex_y']

In [ ]:
all_rmrcodes = list(set(clickup_df['rmrcode'].to_list() + elevate_df['rmrcode'].to_list()))


print(len(all_rmrcodes))

for col in ['plan_year.valid_from', 'plan_year.valid_to']:
    elevate_df[col] = pd.to_datetime(elevate_df[col])


builder = ReportBuilder(elv_df=elevate_df, cu_df=clickup_df)


report_rows = []

for rmrcode in all_rmrcodes:
    row_dict = {}

    row_dict['Organization Name'] = builder.organization_name(rmrcode)
    row_dict['RMRCODE'] = rmrcode

    org_id, client_id = builder.ids(rmrcode)
    row_dict['Organization ID'] = org_id
    row_dict['Client ID'] = client_id

    elv_start, cu_start = builder.start_date(rmrcode)
    row_dict['(Elevate) Plan Year Start Date'] = elv_start
    row_dict['(Clickup) Plan Year Start Date'] = cu_start

    elv_end, cu_end = builder.end_date(rmrcode)
    row_dict['(Elevate) Plan Year End Date'] = elv_end
    row_dict['(Clickup) Plan Year End Date'] = cu_end

    elv_rollover, cu_rollover = builder.rollover(rmrcode)
    row_dict['(Elevate) Rollover/Carryover'] = elv_rollover
    row_dict['(ClickUp) Rollover/Carryover'] = cu_rollover

    elv_rollover_max, cu_rollover_max = builder.rollover_max(rmrcode)
    row_dict['(Elevate) Rollover Max'] = elv_rollover_max
    row_dict['(ClickUp) Rollover Max'] = cu_rollover_max

    elv_carryover_next_year, cu_carryover_net_year = builder.carryover_next_year(rmrcode)
    row_dict['(Elevate) Carryover only through next Plan Year'] = elv_carryover_next_year
    row_dict['(ClickUp) Carryover only through next Plan Year'] = cu_carryover_net_year

    elv_roll_eligibility, cu_roll_eligibility = builder.carryover_if_elect(rmrcode)
    row_dict['(Elevate) Carryover only if elect to participate for next Plan Year'] = elv_roll_eligibility
    row_dict['(ClickUp) Carryover only if elect to participate for next Plan Year'] = cu_roll_eligibility

    elv_hsa_offered, cu_hsa_offered = builder.HSA_offered(rmrcode)
    row_dict['(Elevate) HSA Offered'] = elv_hsa_offered
    row_dict['(ClickUp) HSA Offered'] = cu_hsa_offered

    report_rows.append(row_dict)


report_df = pd.DataFrame(report_rows)



2319


In [23]:
elevate_df[elevate_df['rmrcode'] == "RMRAMA"][['elv_plan_id', 'plan_code', 'account_type.account_type', 'plan_year.valid_from', 'plan_year.valid_to']]

,elv_plan_id,plan_code,account_type.account_type,plan_year.valid_from,plan_year.valid_to
1155,76002,RMRAMADCA0801202507312026,DCAP,2025-08-01,2026-07-31
1165,32774,RMRAMADCA0801202407312025,DCAP,2024-08-01,2025-07-31
1176,76003,RMRAMAFSA0801202507312026,HCFSA,2025-08-01,2026-07-31
1179,51215,RMRAMAHRA0101202512312025,HRA,2025-01-01,2025-07-31
1181,32776,RMRAMAFSA0801202307312024,HCFSA,2023-08-01,2024-07-31
1182,32773,RMRAMADCA0801202307312024,DCAP,2023-08-01,2024-07-31
1191,32777,RMRAMAFSA0801202407312025,HCFSA,2024-08-01,2025-07-31
1200,32775,RMRAMAHRA0101202412312024,HRA,2024-01-01,2024-12-31


In [41]:
report_df2 = report_df.fillna("")
report_df2

,Organization Name,RMRCODE,Organization ID,Client ID,(Elevate) Plan Year Start Date,(Clickup) Plan Year Start Date,(Elevate) Plan Year End Date,(Clickup) Plan Year End Date,(Elevate) Rollover/Carryover,(ClickUp) Rollover/Carryover,(Elevate) Rollover Max,(ClickUp) Rollover Max,(Elevate) Carryover only through next Plan Year,(ClickUp) Carryover only through next Plan Year,(Elevate) Carryover only if elect to participate for next Plan Year,(ClickUp) Carryover only if elect to participate for next Plan Year,(Elevate) HSA Offered,(ClickUp) HSA Offered
0,Harrison School District,RMRHSD,,86877zmv8,,,,,,,,nan,,Not on ClickUp,Idk where to find on elevate,,False,False
1,Town of Monument,RMRMON,8505,86877zw32,01/01,01/01,12/31,12/31,True/False,True/False,"640.0, 660.0","610,640,660,None",True/False,Not on ClickUp,Idk where to find on elevate,,False,False
2,American Academy,RMRAMA,8700,86877zj73,"['01/01', '08/01']","['01/01', '08/01']","['12/31', '07/31']","['12/31', '07/31']",False,False,"640.0, 660.0, 500.0","660,None",True/False/None,Not on ClickUp,Idk where to find on elevate,,False,False
3,Odyssey School,RMRODY,6128,86877zr5j,07/01,07/01,06/30,06/30,True/False,True/False,"640.0, 660.0","610,640,660,None",True/False,Not on ClickUp,Idk where to find on elevate,,False,False
4,Cobb Mechanical Contractors,RMRCOBB,,86877zk7d,,,,,,,,nan,,Not on ClickUp,Idk where to find on elevate,,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2314,Johnson Nathan Strohe PC,RMRJNS,9992,86877znma,12/01,12/01,11/30,11/30,True/False,True/False,"640.0, 660.0","610,640,None",True/False,Not on ClickUp,Idk where to find on elevate,,False,False
2315,Aarista,RMRAAR,13279,868bv83bg,02/01,02/01,01/31,01/31,True/False,True/False,660.0,None,True/False,Not on ClickUp,Idk where to find on elevate,,False,False
2316,Infinity Communities LLC,RMRINF,,8687816xt,,,,,,,,,,Not on ClickUp,Idk where to find on elevate,,False,False
2317,Space Science Institute,RMRSPC,9785,86877zuqx,01/01,01/01,12/31,12/31,False,False,"640.0, 660.0","660,None",True/False,Not on ClickUp,Idk where to find on elevate,,False,False


In [ ]:
rename_map = {

    # clickup # column
    
    "x": "(ClickUp) Plan Year Start Date (Month & Day)",
    "x": "(ClickUp) Plan Year End Date (Month & Day)",
    "y": "(ClickUp) Employer HSA Contributions",                                     # what is this?
    "y": "(ClickUp) Health FSA Limit",                                               # waiting for clarification from ann
    "x": "(ClickUp) Rollover/Carryover",
    "x": "(ClickUp) Rollover Max",
    "y": "(ClickUp) Carryover only through next Plan Year",
    "y": "(ClickUp) Carryover only if elect to participate for next Plan Year",
    "x": "(ClickUp) Limited Purpose Offered",
    "x": "(ClickUp) HSA Offered",
    "": "(ClickUp) Health FSA Grace Period Offered",
    "": "(ClickUp) Health FSA Grace Period Length",
    "": "(ClickUp) Dependent FSA Grace Period Offered",
    "": "(ClickUp) Dependent FSA Grace Period Length",
    "": "(ClickUp) Runout Period Length",
    "": "(ClickUp) Termed Runout Period Length",


    # elv columns

    "": "(Elevate) Plan Year Start Date (Month & Day)",
    "": "(Elevate) Plan Year End Date (Month & Day)",
    "": "(Elevate) Employer HSA Contributions",
    "": "(Elevate) Health FSA Limit",
    "": "(Elevate) Rollover/Carryover",
    "": "(Elevate) Rollover Max",
    "": "(Elevate) Carryover only through next Plan Year",                          # (ClickUp) mark true if rollover is checked yes but rollover eligibility is false
    "": "(Elevate) Carryover only if elect to participate for next Plan Year",      # corresponding clickup field is "rollover eligibility" (any value on clickup maps to true)
    "": "(Elevate) Limited Purpose Offered",
    "": "(Elevate) HSA Offered",
    "": "(Elevate) Health FSA Grace Period Offered",
    "": "(Elevate) Health FSA Grace Period Length",
    "": "(Elevate) Dependent FSA Grace Period Offered",
    "": "(Elevate) Dependent FSA Grace Period Length",
    "": "(Elevate) Runout Period Length",
    "": "(Elevate) Termed Runout Period Length",

}

In [ ]:
column_order = [
    "(Elevate) Plan Year Start Date (Month & Day)",
    "(ClickUp) Plan Year Start Date (Month & Day)",
    "(Elevate) Plan Year End Date (Month & Day)",
    "(ClickUp) Plan Year End Date (Month & Day)",
    "(Elevate) Employer HSA Contributions",
    "(ClickUp) Employer HSA Contributions",
    "(Elevate) Health FSA Limit",
    "(ClickUp) Health FSA Limit",
    "(Elevate) Rollover/Carryover",
    "(ClickUp) Rollover/Carryover",
    "(Elevate) Rollover Max",
    "(ClickUp) Rollover Max",
    "(Elevate) Carryover only through next Plan Year",
    "(ClickUp) Carryover only through next Plan Year",
    "(Elevate) Carryover only if elect to participate for next Plan Year",
    "(ClickUp) Carryover only if elect to participate for next Plan Year",
    "(Elevate) Limited Purpose Offered",
    "(ClickUp) Limited Purpose Offered",
    "(Elevate) HSA Offered",
    "(ClickUp) HSA Offered",
    "(Elevate) Health FSA Grace Period Offered",
    "(ClickUp) Health FSA Grace Period Offered",
    "(Elevate) Health FSA Grace Period Length",
    "(ClickUp) Health FSA Grace Period Length",
    "(Elevate) Dependent FSA Grace Period Offered",
    "(ClickUp) Dependent FSA Grace Period Offered",
    "(Elevate) Dependent FSA Grace Period Length",
    "(ClickUp) Dependent FSA Grace Period Length",
    "(Elevate) Runout Period Length",
    "(ClickUp) Runout Period Length",
    "(Elevate) Termed Runout Period Length",
    "(ClickUp) Termed Runout Period Length"
]